# Dóng hàng theo bài

Sửa khối cấu hình ở đầu cell của tác phẩm rồi chạy bằng kernel `.venv`. Mỗi cell đọc TXT Hán và Việt trong `03_text_sach/`.

1. **Bac Su (CPO_09).** TV04: Han PDF 155-207, Viet PDF 102-148. Can xac dinh ranh gioi tung muc truoc khi dong hang.
2. **Van sach Thi Dinh (CPO_11).** TV04: bai 23-24.
3. **Tho van Ly-Tran (CPO_12).** TV04: bai 314-415.

- `CHAY_HANG_LOAT = False` chạy một bài hoặc mục và hiện bảng xem trước như cũ.
- `CHAY_HANG_LOAT = True` chạy lần lượt các mục trong `DANH_SACH_MUC` hoặc các bài trong `DANH_SACH_BAI`. Có thể chọn vài bài, ví dụ `[314, 315]`. Mỗi bài ghi vào thư mục duyệt riêng.
- `TIEU_DE_THEO_BAI` ghi số dòng đầu cần bỏ ở từng bản. Bài chưa khai báo giữ toàn bộ dòng. Với Lý–Trần, đặt thể loại riêng ở `THE_LOAI_THEO_BAI` khi khác `THE_LOAI`.

- `CHE_DO = 'nguon'` xem trước tối đa 20 dòng mỗi bản, có số dòng. Điền `TU_KHOA` để tìm trong toàn bộ TXT.
- `CHE_DO = 'nhap'` tạo bản nháp và lưu cấu hình vào `draft.json`.
- `CHE_DO = 'xuat'` xuất bản đã duyệt theo cấu hình lưu cùng bản nháp.

Hán–âm dùng code Lab cùng hai từ điển. Hán–nghĩa dùng API CLC và ghi các liên kết vào `draft.json`.

### Dóng dịch thơ qua Codex

1. Chạy `CHE_DO = 'nhap'`. Mỗi bài có bản dịch thơ sẽ sinh `poetry_prompt.json` trong thư mục duyệt in ở output.
2. Đính kèm file `poetry_prompt.json` đó vào chat Codex hoặc dán toàn bộ nội dung file. Dùng yêu cầu sau.

> Đọc `poetry_prompt.json` và ghép bản Hán với bản dịch thơ có sẵn theo ý. Giữ nguyên `work` và `fingerprint`. Trả JSON có `links`, mỗi nhóm gồm `han_ids`, `tho_ids` và `note`. Chỉ dùng ID trong file, giữ đúng thứ tự, mỗi ID dùng tối đa một lần. Bỏ cặp chưa đủ căn cứ.

3. Lưu phần JSON Codex trả về thành `poetry_result.json` trong cùng thư mục với `poetry_prompt.json`. Chạy lại bài đó với `CHE_DO = 'nhap'` để nạp kết quả vào cột Dịch thơ.

Để xuất, sao chép `review_template.json` thành `review.json`, duyệt `links`, ghi lý do trong `skipped` cho từng ID còn sót và đặt `approved` thành `true`. Chạy lại cell của đúng bài hoặc mục với `CHE_DO = 'xuat'`. Kết quả trong `04_dong_hang/` gồm TSV UTF-8, XLSX có sheet `DONG_HANG` và `notes.xlsx`.

**CLC.** Khai báo `CLC_API_KEY` trong `.env` ở thư mục gốc, sau đó chạy cell **Nạp token CLC** bên dưới. Các cell dóng hàng dùng chung token trong kernel. Phản hồi được lưu vào `clc_response.json` trong thư mục duyệt và dùng lại khi đầu vào giống lần trước. Đổi tên file này nếu cần gửi lại cùng đầu vào. [Tài liệu CLC, trang 1 và 11–13](../guideline/TaiLieuAPI_HanNom_CLC.pdf)

Mỗi lượt tạo bản nháp có dịch nghĩa dùng một request. Bộ đếm nghỉ ít nhất 3 giây và giới hạn 25/phút, 50/giờ, 200/ngày, 600/tuần.

API dùng `requests` trong kernel `.venv`.

Thi Dinh: mac dinh bai 24, chay cell nap token truoc. CHE_DO="nhap" xuat ban nhap TSV/XLSX; cac don vi chua dong o cuoi bang. File da ton tai voi noi dung khac se khong bi ghi de.

4. **CP024**: đọc các cặp `NNN.han.txt` / `NNN.viet.txt`; xem hướng dẫn CP024 trong notebooks/README.md.


In [1]:
# Nạp CLC_API_KEY từ .env ở thư mục gốc dự án (dùng chung cho CP024 và các cell khác)
import os
from pathlib import Path


def nap_token_clc():
    here = Path.cwd().resolve()
    project = next((p for p in (here, *here.parents) if (p / 'ten_file.py').is_file()), None)
    if project is None:
        raise FileNotFoundError('Mở notebook trong repository có ten_file.py.')
    env_file = project / '.env'
    if not env_file.is_file():
        raise FileNotFoundError('Không tìm thấy .env tại thư mục gốc dự án. Tạo từ .env.example và khai báo CLC_API_KEY.')
    token = ''
    for line in env_file.read_text(encoding='utf-8-sig').splitlines():
        stripped = line.strip()
        if not stripped or stripped.startswith('#'):
            continue
        key, sep, value = stripped.partition('=')
        if sep and key.strip() == 'CLC_API_KEY':
            token = value.strip()
            if len(token) >= 2 and token[0] == token[-1] and token[0] in ("'", '"'):
                token = token[1:-1]
            break
    if not token or any(character.isspace() for character in token):
        raise ValueError('Khai báo CLC_API_KEY hợp lệ trong .env; không dùng khoảng trắng trong token.')
    os.environ['CLC_API_KEY'] = token
    print('Đã nạp CLC_API_KEY từ .env cho kernel.')

nap_token_clc()


Đã nạp CLC_API_KEY từ .env cho kernel.


In [11]:
# Bac Su TV04 - TXT Han CP0_09_TV04_bai155 ... bai201 + viet.txt
TRANG_DAU = 150
TRANG_CUOI = 202
# Optional paired content ranges, counted from non-empty lines (inclusive).
DONG_HAN = None
DONG_VIET = None
XEM_NGUON = True  # Set False to send to CLC and export TSV/XLSX.

from pathlib import Path
from importlib import reload
import sys
here = Path.cwd().resolve()
project = next((p for p in (here, *here.parents) if (p / "ten_file.py").is_file()), None)
if project is None:
    raise FileNotFoundError("Open notebook inside the repository.")
if str(project) not in sys.path:
    sys.path.insert(0, str(project))
from cong_cu import bac_su_notebook
reload(bac_su_notebook)
bac_su = bac_su_notebook.run(project, first=TRANG_DAU, last=TRANG_CUOI,
                            han_lines=DONG_HAN, viet_lines=DONG_VIET, preview=XEM_NGUON, skip_couplets=True)


Hán 468 đơn vị
CPO_09/TV04/han/0001 十一日修啟稟后道付前路行人阮廷岸陶登誉遊回其礼部及廣西巡振答又名
CPO_09/TV04/han/0002 道一体奉進并有伺寄涼山鎮官轉迎三使為有家書付寄發行四人陷
CPO_09/TV04/han/0003 費銀共六兩打發開上禮土宜六色奉正使八傳落法礼都左侍郎沛川侵陳釋
CPO_09/TV04/han/0004 浚付使八傷添差翰林院傳講頴詠伯黎貴惇傳制会方伯鄭喜附等謹
CPO_09/TV04/han/0005 啓哺皮蓮同司貓御昭衆辟哪差郎役官編產辰腦陋鎮度西省壞相
CPO_09/TV04/han/0006 意象碎色固啟遊衛朝鎮顥近教腦意眾碎進行定節湖度荒早
CPO_09/TV04/han/0007 湘江乾涸腦點近珍買與長沙牆卷翻亂近紅熱漢口腦修而近蔑缤
CPO_09/TV04/han/0008 南京每節首後韵腦獸色總帝時拱特欣又腦期意色斯老天塘進京群
CPO_09/TV04/han/0009 縣官伴送督促股腦近膽修象辟開船溪長江腦近足護殺流黃江
CPO_09/TV04/han/0010 晶近舉致山東省將縣臺兒间河水口凍前遙封祠官地方扒給人夫車焉
CPO_09/TV04/han/0011 卷四辛
CPO_09/TV04/han/0012 貝？
CPO_09/TV04/han/0013 碎象起行陸脑腊终比京官礼部朱吏目具会团馆大使界出三
CPO_09/TV04/han/0014 里外迎接官九行督瓜公馆皇协之目醉駐供帳整潔官
CPO_09/TV04/han/0015 提督素典内礼辑问劳辛善官主客户舒傳报什闲陪至生官连部
CPO_09/TV04/han/0016 轉奏腾进辟進表文奏本尼堂礼部吸然视大红袍進貢表本来
CPO_09/TV04/han/0017 改服官绿税進事表事官礼部右堂后程报浪
CPO_09/TV04/han/0018 光国主服已喻年术必灵服应服大社一体棒进进设象碎技呈文官提
CPO_09/TV04/han/0019 智浪欢邀天恩著護錫封疑望特葛题建俾軒驾男大辟樣拾
CPO_09/TV04/han/0020 仲秋吉戒路仍许貢使先回报知预借迎接以享恭欢欧进官礼部
Việt 790 đơn vị
CPO_09/TV04/nghia/0001 Viên Lại phòng đã trình th

In [9]:
# 2. Văn sách Thi Đình
THANH_VIEN = 'TV04'
CHAY_HANG_LOAT = False
SO_BAI = 24
DANH_SACH_BAI = list(range(23, 25))  # Hoặc chọn [23, 24]
THE_LOAI = 'prose'
TIEU_DE_THEO_BAI = {}
CHE_DO = 'xuat'  # 'nguon', 'nhap' hoặc 'xuat'
TU_KHOA = ''  # Dùng trong chế độ 'nguon'

import sys
from importlib import reload
from pathlib import Path

here = Path.cwd().resolve()
project = next((p for p in (here, *here.parents) if (p / 'ten_file.py').is_file()), None)
if project is None:
    raise FileNotFoundError('Mở notebook trong thư mục project có ten_file.py.')
if str(project) not in sys.path:
    sys.path.insert(0, str(project))

from cong_cu import alignment_backends
reload(alignment_backends)
from cong_cu import dong_hang
reload(dong_hang)
from cong_cu import alignment_notebook
reload(alignment_notebook)

from cong_cu import thi_dinh_notebook
reload(thi_dinh_notebook)

bai_duoc_chon = DANH_SACH_BAI if CHAY_HANG_LOAT else [SO_BAI]
for bai in bai_duoc_chon:
    manifest = thi_dinh_notebook.prepare_sources(project, THANH_VIEN, f'bai{bai:04d}')
    print(f'bai{bai:04d}: {sum(len(item["parts"]) for item in manifest)} TXT')
cau_hinh = [
    {'work': f'bai{bai:04d}', 'genre': THE_LOAI, 'titles': TIEU_DE_THEO_BAI.get(bai)}
    for bai in bai_duoc_chon
]
if CHAY_HANG_LOAT:
    thi_dinh = alignment_notebook.run_book_batch(
        project, 'CPO_11', cau_hinh, member=THANH_VIEN, mode=CHE_DO, keyword=TU_KHOA,
    )
else:
    thi_dinh = alignment_notebook.run_book_cell(
        project, 'CPO_11', **cau_hinh[0], member=THANH_VIEN, mode=CHE_DO, keyword=TU_KHOA,
    )

# Export draft proposals only. The approved export remains CHE_DO='xuat'.
if CHE_DO == 'nhap':
    results = thi_dinh['results'] if CHAY_HANG_LOAT else {f'bai{SO_BAI:04d}': thi_dinh}
    for work, draft in results.items():
        export = thi_dinh_notebook.export_draft(project, THANH_VIEN, work, draft)
        for path in export['files']:
            print(path)
        print('Unmatched:', {k: len(v) for k, v in export['unmatched'].items()},
              '| CLC rejected:', export['rejected'])


bai0024: 49 TXT
CPO_11/bai0024/TV04
Đang xuất bản đã duyệt…
Thư mục duyệt D:\1_Study\3.Year 3\Semester_1\Intro_NLP\Data_processing\dong_hang\han-viet-corpus-toolkit\san_pham\TV04\CV06_CPO_11\05_quan_ly\dong_hang\TV04\bai0024
export | Đã xuất 126 hàng đã duyệt.
D:\1_Study\3.Year 3\Semester_1\Intro_NLP\Data_processing\dong_hang\han-viet-corpus-toolkit\san_pham\TV04\CV06_CPO_11\04_dong_hang\bai0024\CPO_11_bai0024_parallel.tsv
D:\1_Study\3.Year 3\Semester_1\Intro_NLP\Data_processing\dong_hang\han-viet-corpus-toolkit\san_pham\TV04\CV06_CPO_11\04_dong_hang\bai0024\CPO_11_bai0024_parallel.xlsx
D:\1_Study\3.Year 3\Semester_1\Intro_NLP\Data_processing\dong_hang\han-viet-corpus-toolkit\san_pham\TV04\CV06_CPO_11\04_dong_hang\bai0024\notes.xlsx


,Hán,Dịch âm,Dịch nghĩa,Dịch thơ
0,弟二中 阮尚賢 河内山朗連㧞 囯卷臣畫有秤臣𤨔光造奉擕攵理肆为,ĐỆ NHỊ GIÁP TIẾN SĨ XUẤT THÂN NGUYỄN THƯỢNG HI...,ĐỆ NHỊ GIÁP TIẾN SĨ XUẤT THÂN NGUYỄN THƯỢNG HI...,
1,第一名 工酉恩舒次中播不九名陵卷臣叚攵会臣畀为正奉坊乆理得肆分,Thần đối. Thần văn: Minh vương chi cầu ngôn dã...,"Độc quyển: Thần Đoàn Văn Hội, thần Lương Quý C...",
2,臣対臣闻明主之求言也以濟後為急人之福学也以致用為要言不,"Ngôn bất khả dĩ thố vu trị, vị chi hư ngôn. Họ...",Thần xin thưa. Thần nghe: Bậc minh chủ cầu lời...,
3,𫠦以揚于後谓之𠁅墨学不以見诸用理他学故員扆𥘉政防軒盛,"Cố phụ ỷ sơ chính, lâm hiên thịnh điển, quảng ...","Kẻ bề tôi chăm việc học, lấy việc “trí dụng” l...",
4,典廣葑菲云誅異草苐之愚所以求善言而䝨美化也奉我 翼寺英皇帝右道崇夂患土曰之不古学術云不㹠 坡...,Phụng ngã Dực Tôn Anh hoàng đế hữu đạo sùng vă...,"Cho nên khi mới lên ngôi nghe chính sự, cử hàn...",
5,领之中外俾臣者廣𢪷传授母務 哀而為之汝易好諛侫 而偉㕝言以見吏才資𥦗用昭尚弄封策诸人多有学试...,"Vạn cơ chi hạ, phục tiến thần đẳng vu đình, nh...",Cho nên các bài đối sách đương thời phần nhiều...,
6,至衷本朝攵体返過去代飲惟 白弓帝陛下恥彐 以天腦 学洞日新懋迪益光廣𢾾攵敎弟集之暇𣸪進臣另于,"Thần tài sơ thức thiển, học chuyết từ lậu, hạt...",Văn thể bản triều hơn hẳn đời trước. Kính nghĩ...,
7,庭而策以往典表務渚條奉見 聖心方遠塘員古今灰詎仅,Phục độc Chế sách viết: Trẫm duy: Tích minh vư...,"Vâng thấy lòng vua cao xa, xem khắp xưa nay, g...",
8,云陳淡思居辰云雲用臣才踈试𭰦学拙詞陋曷足以上揆渊𠂻,"Cái dĩ cần cầu thượng lý, hoằng tế nguyên nguy...","Thần tài sơ thức mỏng, học kém, lời thô, đâu đ...",
9,雲陳至労飲當腸伊扵敢不勉捜所見以為拜献云先資中仗讀,"Thâm duy phó thác chi trọng, lẫm thủ thành quy...","Nhưng vào buổi đầu đối đáp, dám chẳng gắng góp...",


In [13]:
# 3. Thơ văn Lý–Trần
THANH_VIEN = 'TV04'
CHAY_HANG_LOAT = True
SO_BAI = 314
DANH_SACH_BAI = list(range(314, 416))  # Hoặc chọn [314, 315]
THE_LOAI = 'poetry'  # 'prose' nếu bài là văn xuôi
THE_LOAI_THEO_BAI = {}  # Ví dụ {135: 'prose'} nếu bài đó là văn xuôi
TIEU_DE_THEO_BAI = {}
CHE_DO = 'xuat'  # 'nguon', 'nhap' hoặc 'xuat'
TU_KHOA = ''  # Dùng trong chế độ 'nguon'

import sys
from importlib import reload
from pathlib import Path

here = Path.cwd().resolve()
project = next((p for p in (here, *here.parents) if (p / 'ten_file.py').is_file()), None)
if project is None:
    raise FileNotFoundError('Mở notebook trong thư mục project có ten_file.py.')
if str(project) not in sys.path:
    sys.path.insert(0, str(project))

from cong_cu import alignment_backends
reload(alignment_backends)
from cong_cu import dong_hang
reload(dong_hang)
from cong_cu import alignment_notebook
reload(alignment_notebook)

bai_duoc_chon = DANH_SACH_BAI if CHAY_HANG_LOAT else [SO_BAI]
cau_hinh = [
    {'work': f'bai{bai:04d}', 'genre': THE_LOAI_THEO_BAI.get(bai, THE_LOAI),
     'titles': TIEU_DE_THEO_BAI.get(bai)}
    for bai in bai_duoc_chon
]
if CHAY_HANG_LOAT:
    ly_tran = alignment_notebook.run_book_batch(
        project, 'CPO_12', cau_hinh, member=THANH_VIEN, mode=CHE_DO, keyword=TU_KHOA,
    )
else:
    ly_tran = alignment_notebook.run_book_cell(
        project, 'CPO_12', **cau_hinh[0], member=THANH_VIEN, mode=CHE_DO, keyword=TU_KHOA,
    )



[1/102] bai0314
CPO_12/bai0314/TV04
Đang xuất bản đã duyệt…
Lỗi | Chưa có bản duyệt: D:\1_Study\3.Year 3\Semester_1\Intro_NLP\Data_processing\dong_hang\han-viet-corpus-toolkit\san_pham\TV04\CV06_CPO_12\05_quan_ly\dong_hang\TV04\bai0314\review.json

[2/102] bai0315
CPO_12/bai0315/TV04
Đang xuất bản đã duyệt…
Thư mục duyệt D:\1_Study\3.Year 3\Semester_1\Intro_NLP\Data_processing\dong_hang\han-viet-corpus-toolkit\san_pham\TV04\CV06_CPO_12\05_quan_ly\dong_hang\TV04\bai0315
export | Đã xuất 9 hàng đã duyệt.
D:\1_Study\3.Year 3\Semester_1\Intro_NLP\Data_processing\dong_hang\han-viet-corpus-toolkit\san_pham\TV04\CV06_CPO_12\04_dong_hang\bai0315\CPO_12_bai0315_parallel.tsv
D:\1_Study\3.Year 3\Semester_1\Intro_NLP\Data_processing\dong_hang\han-viet-corpus-toolkit\san_pham\TV04\CV06_CPO_12\04_dong_hang\bai0315\CPO_12_bai0315_parallel.xlsx
D:\1_Study\3.Year 3\Semester_1\Intro_NLP\Data_processing\dong_hang\han-viet-corpus-toolkit\san_pham\TV04\CV06_CPO_12\04_dong_hang\bai0315\notes.xlsx

[3/102] 

In [4]:
# CP024 - Dóng hàng Hán - dịch nghĩa
THANH_VIEN = "TV04"
MA_SACH = "CP024"
CHE_DO = "xuat"  # nguon | nhap | xuat
CHAY_HANG_LOAT = True
SO_FILE = 5
DANH_SACH_FILE = []  # Rỗng khi chạy hàng loạt: tất cả
THE_LOAI = "prose"
PHUONG_PHAP = "clc"  # clc | dp
SO_MUC = None  # Xử lý toàn file
SO_DONG_MOI_MUC = 10  # Tối đa 10 dòng nguồn mỗi bản cho mỗi lượt CLC
TU_KHOA = ""
XAC_NHAN_RANH_GIOI = True
DONG_TOAN_FILE = set()  # Chia mỗi file thành các mục, chạy tất cả mục
KHOANG_DONG_THEO_FILE = {}
THU_MUC_DATA = "data/"  # None: data/; hoặc gốc chứa CP024/
THU_MUC_SAN_PHAM = None
TIEP_TUC_KHI_LOI = False

import sys
from pathlib import Path
here = Path.cwd().resolve()
project = next((p for p in (here, *here.parents) if (p / 'ten_file.py').is_file()), None)
if project is None:
    raise FileNotFoundError('Mở notebook trong repository.')
if str(project / 'src') not in sys.path:
    sys.path.insert(0, str(project / 'src'))
from importlib import reload
from hanviet.alignment.backends import clc_aligner
from hanviet.alignment import runner, cp024 as cp024_module
reload(clc_aligner)
reload(runner)
reload(cp024_module)  # Nạp code mới khi chạy lại cell trong cùng kernel
cp024 = cp024_module.run_cp024_by_work(
    project, member=THANH_VIEN, mode=CHE_DO, batch=CHAY_HANG_LOAT, method=PHUONG_PHAP,
    file_number=SO_FILE, file_numbers=DANH_SACH_FILE, genre=THE_LOAI,
    section_number=SO_MUC, lines_per_section=SO_DONG_MOI_MUC, keyword=TU_KHOA,
    confirmed=XAC_NHAN_RANH_GIOI, whole_files=DONG_TOAN_FILE,
    manual_ranges=KHOANG_DONG_THEO_FILE, data_root=THU_MUC_DATA,
    output_root=THU_MUC_SAN_PHAM, continue_on_error=TIEP_TUC_KHI_LOI,
)


bai0005: 469 hàng
D:\1_Study\3.Year 3\Semester_1\Intro_NLP\Data_processing\dong_hang\han-viet-corpus-toolkit\san_pham\TV04\CV06_CPO_24\04_dong_hang\bai0005\CPO_24_bai0005_parallel.tsv
D:\1_Study\3.Year 3\Semester_1\Intro_NLP\Data_processing\dong_hang\han-viet-corpus-toolkit\san_pham\TV04\CV06_CPO_24\04_dong_hang\bai0005\CPO_24_bai0005_parallel.xlsx
bai0006: 324 hàng
D:\1_Study\3.Year 3\Semester_1\Intro_NLP\Data_processing\dong_hang\han-viet-corpus-toolkit\san_pham\TV04\CV06_CPO_24\04_dong_hang\bai0006\CPO_24_bai0006_parallel.tsv
D:\1_Study\3.Year 3\Semester_1\Intro_NLP\Data_processing\dong_hang\han-viet-corpus-toolkit\san_pham\TV04\CV06_CPO_24\04_dong_hang\bai0006\CPO_24_bai0006_parallel.xlsx


In [ ]:
# 4. Gộp bảng dóng hàng để nộp
THANH_VIEN = "TV04"
TAC_PHAM_CHON = ["CPO_11", "CPO_12"]  # Ví dụ chỉ lấy Thi Đình: ["CPO_11"]

import csv
import io
import re
import sys
from pathlib import Path

here = Path.cwd().resolve()
project = next((p for p in (here, *here.parents) if (p / "ten_file.py").is_file()), None)
if project is None:
    raise FileNotFoundError("Mở notebook trong repo có ten_file.py.")
if str(project) not in sys.path:
    sys.path.insert(0, str(project))

if str(project / "src") not in sys.path:
    sys.path.insert(0, str(project / "src"))
from hanviet.common.paths import resolve_paths
from hanviet.common.io import read_json, write_protected
from hanviet.exporters.tabular import HEADERS, xlsx_bytes
from hanviet.alignment.runner import clean_book_root, load_work
from hanviet.alignment.validation.validators import verify_identity
paths = resolve_paths(user_root=project)

if not TAC_PHAM_CHON or len(set(TAC_PHAM_CHON)) != len(TAC_PHAM_CHON):
    raise ValueError("Chọn ít nhất một mã tác phẩm và mỗi mã chỉ xuất hiện một lần.")
if any(not re.fullmatch(r"(?:CPO_\d{2}|CP024)", code) for code in TAC_PHAM_CHON):
    raise ValueError("Mã tác phẩm phải có dạng CPO_11 hoặc CP024.")

rows = []
table_count = 0
for code in TAC_PHAM_CHON:
    book_root = clean_book_root(project, code, THANH_VIEN, output_root=paths.output_root)
    source = book_root / "04_dong_hang"
    tables = {}
    for extension in ("csv", "tsv"):
        for file in source.glob(f"*/{code}_*_parallel.{extension}"):
            tables[file.with_suffix("")] = file
    if not tables:
        raise FileNotFoundError(f"{code} chưa có bảng dóng hàng đã xuất ở {source}.")

    if code == "CP024":
        audit_root = book_root / "05_quan_ly" / "dong_hang" / THANH_VIEN
        expected = set()
        for plan_file in audit_root.glob("bai*/alignment_plan*.json"):
            plan = read_json(plan_file)
            for section in plan["sections"]:
                expected.add(f"{plan['work']}_{section['section']}")
        actual = {file.parent.name for file in tables.values()}
        full_works = {name.rsplit("_", 1)[0] for name in actual if name.endswith("_muc0000")}
        if any(name.rsplit("_", 1)[0] in full_works and not name.endswith("_muc0000") for name in actual):
            raise ValueError("CP024 có cả bảng toàn file và từng mục của cùng bài; chọn một bộ để tránh gộp trùng.")
        missing = expected - actual
        if missing:
            print("Cảnh báo: mục chưa xuất:", sorted(missing))
        if not expected:
            print("Cảnh báo: chưa có kế hoạch CP024 để kiểm tra đầy đủ.")

    def natural(file):
        return [int(x) if x.isdigit() else x for x in re.split(r"(\d+)", file.as_posix())]
    for file in sorted(tables.values(), key=natural):
        if code == "CP024":
            match = re.fullmatch(r"(bai\d+)_(muc\d+)", file.parent.name)
            if not match:
                raise ValueError(f"Tên mục không hợp lệ: {file}")
            work, section = match.groups()
            plan_path = audit_root / work / ("alignment_plan_toan_file.json" if section == "muc0000" else "alignment_plan.json")
            plan = read_json(plan_path)
            ranges = next((item["ranges"] for item in plan["sections"] if item["section"] == section), None)
            if ranges is None:
                raise ValueError(f"Không có mục trong kế hoạch: {file}")
            _, work_id, _, _, fingerprint = load_work(project, code, work, THANH_VIEN, plan["genre"], {}, section, ranges,
                                                       data_root=paths.data_root, output_root=paths.output_root)
            review = read_json(audit_root / file.parent.name / "review.json")
            verify_identity(review, work_id, fingerprint)
            if review.get("approved") is not True:
                raise ValueError(f"Mục chưa duyệt: {file}")
        with file.open(encoding="utf-8-sig", newline="") as stream:
            if file.suffix == ".tsv":
                work_rows = list(csv.reader(stream, delimiter="\t", quoting=csv.QUOTE_NONE))
            else:
                work_rows = list(csv.reader(stream, strict=True))
                if work_rows and tuple(work_rows[0]) == HEADERS:
                    work_rows = work_rows[1:]
        if not work_rows or any(len(row) != len(HEADERS) for row in work_rows):
            raise ValueError(f"Bảng phải có dữ liệu và đúng 4 cột: {file}")
        rows.extend(work_rows)
    print(f"{code}: {len(tables)} mục / {len({f.parent.name.split('_')[0] for f in tables.values()})} file.")
    table_count += len(tables)

output = paths.output_root / THANH_VIEN / "04_dong_hang"
name = f"{THANH_VIEN}_{'_'.join(TAC_PHAM_CHON)}_parallel"
csv_text = io.StringIO(newline="")
writer = csv.writer(csv_text)
writer.writerow(HEADERS)
writer.writerows(rows)
files = {
    output / f"{name}.csv": csv_text.getvalue().encode("utf-8-sig"),
    output / f"{name}.xlsx": xlsx_bytes("DONG_HANG", HEADERS, rows),
}
write_protected(files)

print(f"Đã gộp {table_count} bảng, tổng {len(rows)} hàng dữ liệu.")
for path in files:
    print(path)
